# Flatland APPO Training on Google Colab


### 1. Install Dependencies


In [ ]:
!pip install flatland-rl
!pip install torch numpy
# After this cell finishes, go to Runtime → Restart runtime, then Run All again

### 2. Create Project Directory Structure


In [ ]:
import os
os.makedirs('agents', exist_ok=True)
open('agents/__init__.py', 'a').close()
print('Directory structure ready.')

### 3. Write Project Files


In [ ]:
%%writefile config.py

class Config:
    # Environment Dimensions
    width  = 50
    height = 50
    n_agents = 12

    # Rail Generation Parameters (Sparse)
    n_cities = 5
    max_rails_between_cities = 3
    max_rail_pairs_in_city   = 2   # 2 pairs = 4 rails per city (max_rails_in_city=4)
    grid_mode = False
    max_episode_steps = 1500
    seed = 1
    
    # Malfunction Parameters
    malfunction_rate         = 0.02
    min_malfunction_duration = 20
    max_malfunction_duration = 50
    
    # Reward Logic
    step_penalty = -0.3
    global_reward = 20.0
    
    # Observation and Action Space
    obs_size = 30   # NewAgentInitObs size
    action_size = 5 # (0: stop, 1: left, 2: straight, 3: right, 4: no-op/logic)
    
    # Training Parameters
    checkpoint_dir = "/content/drive/MyDrive/Flatland/checkpoints"
    lr = 1e-4
    gamma = 0.99
    batch_size = 64
    alpha = 0.2 # Temperature parameter for SAC (if still used elsewhere)
    memory_size = 1000000 # Standard MARL buffer capacity
    
    # PPO Parameters
    ppo_epochs = 4
    ppo_clip = 0.2
    entropy_coef = 0.01
    
    # Coordination / Expert Logic Toggle
    use_expert = True


In [ ]:
%%writefile shortestpath.py
import math
from typing import Dict, List, Optional, Tuple, Set

import numpy as np

from flatland.core.grid.grid4 import Grid4TransitionsEnum
from flatland.core.grid.grid4_utils import get_new_position
from flatland.core.transition_map import GridTransitionMap
from flatland.envs.step_utils.states import TrainState
from flatland.envs.rail_env import RailEnvActions
from flatland.envs.rail_trainrun_data_structures import Waypoint
from collections import namedtuple

# RailEnvNextAction was removed in Flatland 4.x — define it locally.
RailEnvNextAction = namedtuple('RailEnvNextAction', ['action', 'next_position', 'next_direction'])

# Flatland 4.x active states (agent is on the grid)
_ACTIVE_STATES = {TrainState.MOVING, TrainState.STOPPED, TrainState.MALFUNCTION}

# ---------------------------------------------------------------------------
# Rail transition cache  (per worker process — safe with mp spawn)
# Key: (position, direction)  Value: transitions tuple
# Must be cleared on every env.reset() because the rail layout regenerates.
# ---------------------------------------------------------------------------
_trans_cache: dict = {}

def clear_trans_cache() -> None:
    """Clear the rail transition cache. Call after every env.reset()."""
    _trans_cache.clear()


def get_valid_move_actions_(agent_direction: Grid4TransitionsEnum,
                            agent_position: Tuple[int, int],
                            rail: GridTransitionMap) -> Set[RailEnvNextAction]:
    """Get the valid move actions (forward, left, right) for an agent."""
    valid_actions: Set[RailEnvNextAction] = set()
    _key = (agent_position, agent_direction)
    if _key not in _trans_cache:
        _trans_cache[_key] = rail.get_transitions((agent_position, agent_direction))
    possible_transitions = _trans_cache[_key]
    num_transitions = np.count_nonzero(possible_transitions)

    if rail.is_dead_end(agent_position):
        action = RailEnvActions.MOVE_FORWARD
        exit_direction = (agent_direction + 2) % 4
        if possible_transitions[exit_direction]:
            new_position = get_new_position(agent_position, exit_direction)
            valid_actions.add(RailEnvNextAction(action, new_position, exit_direction))
    elif num_transitions == 1:
        action = RailEnvActions.MOVE_FORWARD
        for new_direction in [(agent_direction + i) % 4 for i in range(-1, 2)]:
            if possible_transitions[new_direction]:
                new_position = get_new_position(agent_position, new_direction)
                valid_actions.add(RailEnvNextAction(action, new_position, new_direction))
    else:
        for new_direction in [(agent_direction + i) % 4 for i in range(-1, 2)]:
            if possible_transitions[new_direction]:
                if new_direction == agent_direction:
                    action = RailEnvActions.MOVE_FORWARD
                elif new_direction == (agent_direction + 1) % 4:
                    action = RailEnvActions.MOVE_RIGHT
                elif new_direction == (agent_direction - 1) % 4:
                    action = RailEnvActions.MOVE_LEFT
                else:
                    continue
                new_position = get_new_position(agent_position, new_direction)
                valid_actions.add(RailEnvNextAction(action, new_position, new_direction))
    return valid_actions


def get_shortest_paths(distance_map,
                       agent_pos=None,
                       agent_dir=None,
                       max_depth: Optional[int] = None,
                       agent_handle: Optional[int] = None) -> Dict[int, Optional[List[Waypoint]]]:
    """
    Computes the shortest path for each agent to its target.
    Uses Flatland 4.x TrainState API (flatland.envs.step_utils.states).
    """
    shortest_paths = dict()

    def _shortest_path_for_agent(agent, agent_pos, agent_dir):
        # Flatland 4.x uses agent.state (TrainState enum)
        state = agent.state

        if agent_pos is None:
            if state == TrainState.WAITING:
                # Agent not yet on grid — start from initial position
                position  = agent.initial_position
                direction = agent.initial_direction
            elif state in _ACTIVE_STATES:
                position  = agent.position
                direction = agent.direction
            elif state == TrainState.DONE:
                position  = agent.target
                direction = agent.direction
            else:
                # MALFUNCTION_OFF_MAP or unknown — skip
                shortest_paths[agent.handle] = None
                return
        else:
            position  = agent_pos
            direction = agent_dir

        shortest_paths[agent.handle] = []
        distance = math.inf
        depth = 0

        while position != agent.target and (max_depth is None or depth < max_depth):
            next_actions = get_valid_move_actions_(direction, position, distance_map.rail)
            best_next_action = None

            for next_action in next_actions:
                next_action_distance = distance_map.get()[
                    agent.handle,
                    next_action.next_position[0],
                    next_action.next_position[1],
                    next_action.next_direction
                ]
                if next_action_distance < distance:
                    best_next_action = next_action
                    distance = next_action_distance

            shortest_paths[agent.handle].append(Waypoint(position, direction))
            depth += 1

            if best_next_action is None:
                shortest_paths[agent.handle] = None
                return

            position  = best_next_action.next_position
            direction = best_next_action.next_direction

        if max_depth is None or depth < max_depth:
            shortest_paths[agent.handle].append(Waypoint(position, direction))

    if agent_handle is not None:
        _shortest_path_for_agent(distance_map.agents[agent_handle], agent_pos, agent_dir)
    else:
        for agent in distance_map.agents:
            _shortest_path_for_agent(agent, agent_pos, agent_dir)

    return shortest_paths


In [ ]:
%%writefile NewAgentInitObs.py
from __future__ import division

import copy
from operator import itemgetter

import numpy as np
from flatland.core.env_observation_builder import ObservationBuilder
#from flatland.envs.rail_env_shortest_paths import get_shortest_paths
from shortestpath import get_shortest_paths 

class StateMaskingObs(ObservationBuilder):
    ENTRYS_PER_COLUMN = 8
    TRAFFIC_LIGHT_SIZE = 3
    HOMO_SIZE = 3
    OBS_SIZE = 3 * ENTRYS_PER_COLUMN + TRAFFIC_LIGHT_SIZE + HOMO_SIZE
    ADDITIONAL_INPUT = OBS_SIZE - 3 * ENTRYS_PER_COLUMN

    def __init__(self):
        super(StateMaskingObs, self).__init__()
        self.fake_envs = []
        self.single_solver = []
        self.SKIPLARGE = True 

    def set_env(self, env):
        super().set_env(env)
        
    def reset(self):
        self.time = 0
        self.junctions = []
        self.visited = []
        self.actual_stopping_positions = []
        self.stopping_positions_only = []
        self.actual_junction_cluster = []
        self.permanent_pointer_position = []
        self.temporary_pointer_position = []
        self.initialize_list = [0 for i in range(len(self.env.agents))]
        self.num_agent = len(self.env.agents)
        self.agent_in_clusters = [[-1, -1] for i in range(len(self.env.agents))]
        self.num_active_agents = [0 for i in range(len(self.env.agents))]
        self.initialization_timestep = 0
        self.max_timestep = int((8 * (self.env.height + self.env.width)) / len(self.env.agents))
        self.upper_bound = int((self.env.height + self.env.width) / 12)
        self.observations = [0 for i in range(len(self.env.agents))]
        self.queues = {}
        self.agents_activated = []
        self.clusters_activated = []
        self.path_dict= {}
        self.State ={}
        self.Next_Positions = {}
        self.agents_stuck = [[0,0] for i in range(len(self.env.agents))]
        self.old_info = [ [0,0 ] for i in range(len(self.env.agents))]
        # --- Transition caches (cleared each episode; rail regenerates on reset) ---
        self._rail_trans_cache = {}   # (pos, direction) -> transitions tuple
        self._total_trans_cache = {}  # position         -> total transition count
        return

    def _cheat_expert(self, start_pos, orientation, agentID):
        """
        return the next position when expert is standing on a junction
        """
        target = self.env.agents[agentID].target
        agent_inform = (start_pos,orientation,target)
        if agent_inform in self.path_dict.keys() :
            return self.path_dict[agent_inform] 
        else :
            path = get_shortest_paths(self.distance_map,start_pos,orientation, agent_handle = agentID)
            self.path_dict[agent_inform] = path[agentID] 
            return path[agentID]

    def get_distance_map(self):
        self.distance_map = self.env.distance_map
        if hasattr(self.distance_map, 'reset'):
            self.distance_map.reset(self.env.agents, self.env.rail)
        return   

    def get(self, handle=0):
        """
        param-handle: agent id
        if agent_id==0, add Obs of all agents to self.observations,
        return respective Obs of agent_id
        New obs is a 3*8+3+3 tuple observation for RL
        """

        def is_junction_homo(cell_list):
            assert len(cell_list) == 3
            homo_output = [0, 0, 0]
            count_dict = {}
            junction_pos = None
            # index = []
            for i in cell_list:
                index = [x for x in range(len(cell_list)) if cell_list[x] == i]
                # for x in range(len(cell_list)):
                #     if cell_list[x] is not None:
                #         print(cell_list[x], i)
                #         if cell_list[x] == i:
                #             index.append(x)
                count_dict.update({i: index})
            if 1 <= len(count_dict) <= 3:
                if len(count_dict) == 1:
                    k = list(count_dict.keys())
                    if k[0] is not None:
                        homo_output = [1, 1, 1]
                        junction_pos = k[0]
                elif len(count_dict) == 2:
                    for key, value in count_dict.items():
                        if len(value) == 2 and key is not None:
                            junction_pos = key
                            for index in value:
                                homo_output[index] = 1
            else:
                raise RuntimeError('bug somewhere')
            return homo_output

        def initialize_stopping_points():
            """
            Compute all junction clusters and the stopping points associated with them
            Only needs to be called once for an episode
            """
            self.compute_all_junctions()
            self.compute_stopping_points()
            self.set_stopping_pointers()

        #    if self.initialize_list[handle] == 1:
        #        self.agent_initial_positions[handle] = [0, (-3, -3)]
        if len(self.env.agents) >81 and self.SKIPLARGE == True :
            if ((len(self.env.agents)==100) and ((self.env.height+self.env.width)!=200)) or len(self.env.agents) >100   :
                if self.time == 0 :
                    for agent in range(len(self.env.agents)) :
                        self.observations[agent] = [0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                        0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                        0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                        0.0, 0.0, 0.0,  
                                                        1.0, 0.0, -1.0]
                self.time+=1                                         
                return                                     
        if handle == 0:
            self.get_others_complete() 
            if self.time == 0:
                self.GetAllStates() 
                initialize_stopping_points()
                #self.get_initial_positions()
                self.get_initialization_queue()
                self.get_distance_map() 
            self.update_pointers()
            # Get clusters with stuck agents
            self.get_stuck_clusters()
            self.get_timed_out_clusters()
            # Increment Time
            self.time += 1
            self.initialization_timestep +=1 
            for agent in range(len(self.env.agents)) :
                
                my_pos = self.env.agents[agent].position if self.env.agents[agent].position is not None else self.env.agents[
                    agent].initial_position
                my_direction = self.env.agents[agent].direction if self.env.agents[agent].direction is not None else \
                    self.env.agents[
                        agent].initial_direction
                
                if self.time >2 and not self.env.dones[agent] and self.agents_stuck[agent][0] ==0 and agent in self.agents_activated:
                    if self.old_info[agent][0] == my_pos and self.old_info[agent][1] == my_direction:
                        self.agents_stuck[agent][1] +=1 
                    else :
                        self.agents_stuck[agent][1] = 0     

                self.old_info[agent][0] = copy.deepcopy(my_pos)
                self.old_info[agent][1] = copy.deepcopy(my_direction)         
                
                if self.agents_stuck[agent][1] >100 or self.agents_stuck[agent][0] == 1 and not self.env.dones[agent] :
                    self.agents_stuck[agent][0] = 1  
                    self.observations[agent] = [0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, 0.0, 0.0,  # homogenous junction cell
                                                1.0, 0.0, -1.0]
                    continue                             
                

                if self.env.dones[agent]:
                    self.num_active_agents[agent] = 2
                    self.observations[agent] = [0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                                                0.0, 0.0, 0.0,  # homogenous junction cell
                                                1.0, 0.0, -1.0]
                    continue  # last element: traffic light            
                    
                state_of_agent = self.StateClassifier(my_pos,my_direction) 
                if state_of_agent in [0,3,4] and agent in self.agents_activated :
                    self.observations[agent] = [0.0, -1.0, 0.0, 0.0, 0.0 ,0.0, 0.0, 0.0,
                            0.0, -1.0, 0.0, 0.0, 0.0 ,0.0, 0.0, 0.0,
                            0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.0, 0  , -1 ,  # homagogenous junction cell
                            1.0, 0.0, -1.0]
                    continue         

                isStoppingPoint = False        

                actual_pos = self.env.agents[agent].position if self.env.agents[agent].position is not None else \
                    self.env.agents[
                        agent].initial_position
                actual_dir = self.env.agents[agent].direction if self.env.agents[agent].direction is not None else \
                    self.env.agents[
                        agent].initial_direction

                # Get Traffic Signal
                if agent in self.agents_activated: 
                    others_traffic = self.get_others(agent,1)
                    traffic_signal = self.get_traffic_signal(actual_pos, actual_dir, others_traffic, agent)
                else :
                    traffic_signal = 0     

                obs = []
                others = []
                next_junction_list = []
                all_handles = [i for i in range(len(self.env.agents))]


                # Adjust Position if at Stopping Point 
                if state_of_agent in [1, 4]:
                    valid_act_pos_pair = self._get_next_valid_position(my_pos, my_direction)
                    for action, next_pos in valid_act_pos_pair.items():
                        if next_pos[1] is not None:
                            my_pos = next_pos[1]
                            my_direction = next_pos[0]
                            isStoppingPoint = True
                            break

                # Check whether agent is at decision state 
                if state_of_agent ==2 :
                    state_of_agent = 1
                else:
                    state_of_agent = -1
                
                valid_act_pos_pair = self._get_next_valid_position(my_pos, my_direction)
                for action, next_pos in valid_act_pos_pair.items():
                    # stopping = self.stopping_point_occupied(actual_pos, actual_dir, my_pos, my_direction, action, others)
                    if next_pos[1] is not None:  # valid action
                        # print(next_pos)
                        has_solution, obs_1_direction, junction_cell = self._count_path_block(next_pos[1], next_pos[0],
                                                                                              agent,
                                                                                              my_pos,
                                                                                              my_direction)
                        next_junction_list.append(junction_cell)
                        if has_solution:

                            obs_1_direction[0] += 1  # compensate 2 cell for current pos

                            if isStoppingPoint:
                                #    obs_1_direction[6] = 1
                                obs_1_direction[
                                    0] += 1  # compensate 1 cell for stopping point bc it receives future obs
                            obs_1_direction.insert(0, True)
                            obs.append(obs_1_direction)
                        else:
                            # print(my_pos, my_direction, self.StateClassifier(my_pos, my_direction), "still fail!")
                            obs_1_direction.insert(0, False)
                            obs.append(obs_1_direction)
                    else:
                        next_junction_list.append(None)
                        obs.append([False, 0, 0, 0, 0, 0, 0, 0])
                obs = np.array(obs, dtype=float)
                max_length = max(obs[:, 1])
                for i in range(3):
                    obs[i, 1] = obs[i, 1] / max_length if obs[i, 1] > 0 else -1
                obs = np.reshape(obs, (1, -1))
                obs = obs.squeeze()
                obs = obs.tolist()
                assert len(obs) == 3 * self.ENTRYS_PER_COLUMN

                obs.append(traffic_signal)

                if agent in self.agents_activated:
                    obs.append(0)
                else:
                    obs.append(1)

                obs.append(state_of_agent)
                if agent in self.agents_activated: 
                    junction_homo = is_junction_homo(next_junction_list)
                    obs.extend(junction_homo)
                else :
                    junction_homo = [0,0,0] 
                    obs.extend(junction_homo)  
                      
                assert len(junction_homo) == self.HOMO_SIZE
                assert len(obs) == self.OBS_SIZE
                self.observations[agent] = obs

                # initialization_code()  
            self.get_initialization()
            return self.observations[0]

        else:
            return self.observations[handle]

    def get_many(self, handles=None):
        observations = {}
        self.get(0)  # store all obs in self.observation
        if handles is None:
            handles = []
        for h in handles:
            observations[h] = self.observations[h]
        return observations

    def _get_transitions(self, pos, direction):
        """Cached wrapper around rail.get_transitions."""
        key = (pos, direction)
        if key not in self._rail_trans_cache:
            self._rail_trans_cache[key] = self.env.rail.get_transitions((pos, direction))
        return self._rail_trans_cache[key]

    def StateComputation(self,agent_pos,agent_dir) :
        avb_moves = self._get_transitions(agent_pos, agent_dir)
        move2grid = np.array([[[0, -1], [-1, 0], [0, +1]], [[-1, 0], [0, +1], [+1, 0]], [[0, +1], [+1, 0], [0, -1]],
                              [[+1, 0], [0, -1], [-1, 0]]])  # Obtained from colliding agent code
        trans2act = np.array([[2, 3, 0, 1], [1, 2, 3, 0], [0, 1, 2, 3], [3, 0, 1, 2]])  # Maps transition to an action
        # next_dir_grid = np.array([-1,0,1])  # Maps action to a change in agent direction
        if sum(avb_moves) > 1:  # This is definitely a decision junction since more than 1 move possible
            self.State[agent_pos,agent_dir] = [2,None] 
            self.Get_Valid_Positions(agent_pos,agent_dir,2)  
            return 
        elif sum(avb_moves) == 1:
            avbmove = avb_moves.index(1)  # Get the available transition to next cell
            action = trans2act[agent_dir][avbmove]  # Get the corresponding action for that transition
            if action == 0:
                next_pos = agent_pos + move2grid[(agent_dir + 2) % 4][
                    1]  # This is a dead end, so turn around and move forward
            else:
                next_pos = agent_pos + move2grid[agent_dir][action - 1]    
            # next_dir = (agent_dir + (next_dir_grid[action-1]) )%4
            sumnextcell = 0  # How many possible transitions at next cell
            for i in range(0, 4):
                new_avb_moves = self._get_transitions(tuple(next_pos), i)
                sumnextcell += sum(new_avb_moves)
            if (sumnextcell > 2) :
                self.State[agent_pos,agent_dir] = [1,tuple(next_pos)] 
                self.Get_Valid_Positions(agent_pos,agent_dir,1) 
                return
            elif (sumnextcell <= 2) :
                self.State[agent_pos,agent_dir] = [0,tuple(next_pos)] 
                self.Get_Valid_Positions(agent_pos,agent_dir,0)  
                return 
        else:
            self.State[agent_pos,agent_dir] = [-1,None]
            next_positions = {}
            for action in [1, 2, 3]:
                next_positions.update({action: [None, None]}) 
            self.Next_Positions[agent_pos,agent_dir] = next_positions     
            return

    def GetAllStates(self):
        for row in range(self.env.height):
            for column in range(self.env.width):
                position = (row, column)
                for direction in range(0,4) :
                    self.StateComputation(position,direction) 
        return             
                
    def StateClassifier(self, agent_pos, agent_dir):
        """
        returns 0 : No decision point
        returns 1 : Stopping point (Decision at next cell)
        returns 2 : At decision point currently (More than 1 available transition)
        returns 3,4 : MUST STOP point - Agent Ahead
        returns None: invalid cell
        """
        output = self.State[agent_pos,agent_dir]
        state = output[0]
        next_position = output[1] 
        if state ==2 :
            return 2 
        elif state ==0 :
            others = self.get_others(0,2) 
            if next_position in others:
                return 3
            else :
                return 0 
        elif state ==1 :
            others = self.get_others(0,2) 
            if next_position in others:
                return 4
            else :
                return 1
        else : 
            return None                  

    def Get_Valid_Positions(self, my_pos, my_direction,state):
        """
        action: 0 ---> stop
                1 ---> left
                2 ---> forward
                3 ---> right
        """
        avb_moves = self.env.rail.get_transitions((my_pos, my_direction))
        action2direction = [[3, 0, 1], [0, 1, 2], [1, 2, 3], [2, 3, 0]]
        dir2grid = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
        next_dir_grid = np.array([-1, 0, 1])
        move2grid = np.array([[[0, -1], [-1, 0], [0, +1]],
                              [[-1, 0], [0, +1], [+1, 0]],
                              [[0, +1], [+1, 0], [0, -1]],
                              [[+1, 0], [0, -1], [-1, 0]]])
        avbmove = [i for i, x in enumerate(avb_moves) if x == 1]
        trans2act = np.array([[2, 3, 0, 1], [1, 2, 3, 0], [0, 1, 2, 3], [3, 0, 1, 2]])  # Maps transition to an action
        next_positions = {}
        for action in [1, 2, 3]:
            next_positions.update({action: [None, None]})

        if state in [0, 1, 2, 3, 4]:
            if state == 2:  # decision point
                for action in [1, 2, 3]:
                    i = action2direction[my_direction][action - 1]
                    if i in avbmove:  # available NSWE direction
                        next_pos = my_pos + dir2grid[i]
                        next_positions[action] = [i, tuple(next_pos)]

            else:
                avbmove = avb_moves.index(1)  # Get the available transition to next cell
                action = trans2act[my_direction][avbmove]  # Get the corresponding action for that transition
                if action == 0:
                    next_dir = (my_direction + 2) % 4
                    next_pos = my_pos + move2grid[next_dir][1]
                    # This is a dead end, so turn around and move forward
                else:
                    next_pos = my_pos + move2grid[my_direction][action - 1]
                    next_dir = (my_direction + (next_dir_grid[action - 1])) % 4
                if action == 2 or action == 0 or sum(avb_moves) == 1:
                    next_positions[2] = [next_dir, tuple(next_pos)]
                else:
                    next_positions[action] = [next_dir, tuple(next_pos)]
        self.Next_Positions[my_pos,my_direction] = next_positions
        return next_positions

    def _get_next_valid_position(self,my_pos,my_direction):
        return self.Next_Positions[my_pos,my_direction]   

    def DistToNextJunction(self, agentID, full_path, old_pos, old_heading):
        """
        Returns 1 : If at Stopping Point 
        Returns Distance to Junction (Greater than 1) : If at No Decision Point
        Returns 0 : If at Junction currently 
        """
        full_path_cp = copy.deepcopy(full_path)
        state = self.StateClassifier(old_pos, old_heading)
        sumcell = 0  # How many possible transitions at next cell
        # for j in range(0, 4):
        #     new_avb_moves = self.env.rail.get_transitions(*old_pos, j)
        #     sumcell += sum(new_avb_moves)
        if state in [1, 4]:
            return 1
        elif state in [0, 2, 3]:
            distance = 0
            if state == 2:
                full_path_cp.pop(0)  # remove current junction pos, and add 1 more step
                distance += 1
            for i in range(1, len(full_path_cp) - 1):  # full_path_cp[0] is current pos, not moving yet
                distance += 1
                statecell = self.StateClassifier(full_path_cp[i].position, full_path_cp[i].direction)
                if statecell in [1, 4]:
                    return distance + 1
                elif statecell == 2:
                    return distance
            return distance
        else:
            print("Some error in DistToNextJunction")
            return 0

    def check_transition_validity(self, next_pos, current_pos):
        directions = [-1, -1, -1, -1]
        current_pos = tuple(current_pos)
        if (current_pos[0] >= 0 and 0 <= current_pos[1] < self.env.width and current_pos[
            0] < self.env.height):
            for i in range(0, 4):
                next_positions = self._get_next_valid_position(tuple(current_pos), i)
                for j in range(1, 4):
                    if next_positions[j][1] == next_pos:
                        directions[i] = i
                        break
            if sum(directions) >= -3:
                return True, directions
            else:
                return False, directions
        else:
            return False, directions

    def stopping_point_occupied(self, actual_pos, actual_dir, current_pos, current_dir, action, others):
        action = action - 1
        # print(others)
        count = 0

        movegrid = np.array([[[0, -1], [-1, 0],
                              [0, +1], [+1, 0]],
                             [[-1, 0], [0, +1],
                              [+1, 0], [0, -1]],
                             [[0, +1], [+1, 0],
                              [0, -1], [-1, 0]]])
        if self.StateClassifier(actual_pos, actual_dir) == 2:
            next_pos = actual_pos + movegrid[action][actual_dir]
            validity, directions = self.check_transition_validity(actual_pos, next_pos)
            if validity:
                for i in range(0, 4):
                    if [tuple(next_pos.reshape(1, -1)[0]), directions[i]] in others:
                        return 1
            return 0

        elif self.StateClassifier(actual_pos, actual_dir) in [1, 4]:
            next_pos = current_pos + movegrid[action][current_dir]
            validity, directions = self.check_transition_validity(current_pos, next_pos)
            if validity:
                for i in range(0, 4):
                    if [tuple(next_pos.reshape(1, -1)[0]), directions[i]] in others:
                        return 1
            return 0

        elif self.StateClassifier(actual_pos, actual_dir) in [0, 3] and action == 1:
            new_pos = actual_pos
            new_dir = actual_dir
            while self.StateClassifier(new_pos, new_dir) not in [1, 2, 4]:
                next_positions = self._get_next_valid_position(new_pos, new_dir)
                new_dir = next_positions[2][0]
                new_pos = next_positions[2][1]
                count += 1
                if count > 30:
                    return 0
            for i in range(0, 4):
                if [new_pos, i] in others:
                    return 1
            return 0
        elif self.StateClassifier(actual_pos, actual_dir) in [0, 3] and action in [0, 2]:
            return 0
        else:
            return 0

    def isJuntion(self, agent_pos):
        trans = []
        for direction in range(0, 4):
            trans.append(sum(self.env.rail.get_transitions((agent_pos, direction))))
        return sum(trans) > 2

    def _count_path_block(self, start_pos, heading, agentID, old_pos, old_heading):
        """
        input: the start position (x,y).
        make sure the start position is a rail cell not a obstacle!
        Convert a list of directions to the opposite direction list.
        return: a bool and a 6-element list
                int: does_this_direction has expert solution?,
                [int: astar_path length,
                 int:num_blocking within the first junction,
                 int:num_all_blocking,
                 int:num_blocking on junctions,
                 int: distance to the next junction]
        """

        def next_junction_cell(full_path):
            for i in range(len(full_path)):
                check_pos = full_path[i]
                if self.isJuntion(check_pos.position):
                    return full_path[i].position
            return None

        assert self.env.rail.grid[(start_pos[0], start_pos[1])] != 0, "start position " + str(start_pos) \
                                                                      + " is not valid in the map!"

        count_first_decision_block = 0
        count_first_junction_block = 0
        count_all_block = 0
        count_all_decision_block = 0
        count_decision_point = 0
        count_junction = 0
        crash_buffer = []
        visit_first_decision_point = False
        visit_first_stopping = False

        actual_pos = self.env.agents[agentID].position if self.env.agents[agentID].position is not None else \
            self.env.agents[
                agentID].initial_position
        actual_direction = self.env.agents[agentID].direction if self.env.agents[agentID].direction is not None else \
            self.env.agents[
                agentID].initial_direction

        all_handle = [i for i in range(len(self.env.agents))]

        others_pos = self.get_others(agentID,3) 
        others_directions = self.get_others(agentID,4) 
        #others_moving = []

        # start to compute quantities ---------------------------------------
        full_path = self._cheat_expert(start_pos, heading, agentID)

        if full_path is None:
            return False, [-1,  # single agent path length
                           min(1, count_first_decision_block),  # num_block within the first junction
                           min(1, count_first_junction_block),
                           count_first_decision_block,
                           count_all_block / self.num_agent,  # num_block all along the path
                           count_all_decision_block,  # num_block standing on junctions
                           -1], None

        distance_to_next_junction = self.DistToNextJunction(agentID, full_path, actual_pos,
                                                            actual_direction)
        junction_cell = next_junction_cell(full_path)

        for num_step in range(len(full_path) - 1):
            checking_cell = full_path[num_step].position  # checking cell is the cell we want to check blocking
            checking_cell_dir = full_path[num_step].direction

            if self.isJuntion(checking_cell):
                count_junction += 1
            if self.StateClassifier(checking_cell, checking_cell_dir) == 2:
                count_decision_point += 1

            for direction in range(4):
                if self.StateClassifier(checking_cell, direction) in [1, 4] and not visit_first_stopping:
                    visit_first_stopping = True

            if (checking_cell in others_pos.keys()) is True:
                idx = others_pos[checking_cell]
                crash_buffer.append(checking_cell)
                # test if there is other agents stepping on the stopping point

                if others_directions[idx] == checking_cell_dir:
                    # same heading, not moving, so waiting
                    if num_step == 0:
                        count_first_decision_block += 1

                elif (others_directions[idx] + checking_cell_dir) % 2 == 0:
                    # opposite heading, so blocking
                    count_all_block += 1
                    if self.StateClassifier(checking_cell, checking_cell_dir) == 2:
                        count_all_decision_block += 1

                    if not visit_first_decision_point:
                        count_first_decision_block += 1
                    if not visit_first_stopping:  # stopping point must lead to a junction
                        count_first_junction_block += 1
                else:  # neither same direction or opposite direction, meaning that an agent staying at a junction
                    # but that is not a junction for the current direction (non-decision point)
                    count_all_block += 1
                    count_all_decision_block += 1
                    if not visit_first_decision_point:
                        count_first_decision_block += 1

            if self.StateClassifier(checking_cell, checking_cell_dir) == 2:
                visit_first_decision_point = True

        return True, [len(full_path),  # single agent path length
                      min(1, count_first_decision_block),  # num_block within the first decision point
                      min(1, count_first_junction_block),  # num_block within the first junction
                      ((count_first_decision_block - count_first_junction_block) / max(1, count_first_decision_block)),
                      count_all_block / self.num_agent,  # num_block all along the path
                      count_all_decision_block / count_decision_point if count_decision_point else 0,
                      # num_block standing on junctions
                      distance_to_next_junction / len(full_path) if len(full_path) > 0 else -1,
                      ], junction_cell

    def compute_all_junctions(self):
        """
        Finds all junctions : Any cell with greater than 2 transitions available
        """
        for row in range(self.env.height):
            for column in range(self.env.width):
                position = (row, column)
                if self.total_transitions(position) > 2:
                    self.junctions.append(position)
        return

    def total_transitions(self, position):
        """
        Input - Position
        Return- Total transitions available at a particular cell location
        Called by compute_all_junctions and visit (recursive) — cached for speed.
        """
        if position in self._total_trans_cache:
            return self._total_trans_cache[position]
        sum_transitions = 0
        for i in range(0, 4):
            sum_transitions += sum(self._get_transitions(position, i))
        self._total_trans_cache[position] = sum_transitions
        return sum_transitions

    def compute_stopping_points(self):
        """
        Input - Environment
        Return - None
        Computes all stopping points and junction clumps and stores them
        """
        for position in self.junctions:  # Iterate over all junctions
            if position not in self.visited:  # Only visit junctions which have not been visited before
                self.visited.append(position)  # Keep track of junctions visited
                self.stopping_points = []
                self.stopping_points_only = []
                self.junction_cluster = []
                self.junction_cluster.append(position)  # Computing Junction clusters
                self.visit(position)  # VISIT the current junction
                self.actual_stopping_positions.append(self.stopping_points)
                self.actual_junction_cluster.append(self.junction_cluster)
                self.stopping_positions_only.append(self.stopping_points_only)
        return

    def get_possible_positions(self, position):
        """
        Input - Position
        Returns - 4 Positions which can be reached by this cell , doesn't check for validity of the transition
        """

        movements = np.array([[[-1, 0], [0, +1],
                               [1, 0], [0, -1]]])
        possible_positions = []
        for i in range(0, 4):
            next_pos = position + movements[0][i]
            next_pos = tuple(next_pos.reshape(1, -1)[0])
            possible_positions.append(next_pos)
        return possible_positions

    def visit(self, position):
        """
        Recursive code to visit a junction position and compute the junction clump
        """
        possible_stopping_points = self.get_possible_positions(
            position)  # Get possible positions from the current position
        for stopping_position in possible_stopping_points:
            if (stopping_position is not None):
                valid, directions = self.check_transition_validity(position,
                                                                   stopping_position)  # Check if transition to that cell is possible
                if not valid:
                    continue
                else:
                    for j in range(0, 4):
                        if (0 < self.total_transitions(stopping_position) <= 2) and (
                                (stopping_position, directions) not in self.stopping_points) and (
                                directions[j] != -1):  # Check whether that cell is a stopping point
                            self.stopping_points.append(
                                (stopping_position, directions[j]))  # which has not been visited yet
                            self.stopping_points_only.append(stopping_position)
                        elif self.total_transitions(stopping_position) > 2 and (
                                stopping_position not in self.visited):  # Check whether that cell is a junction
                            self.junction_cluster.append(stopping_position)  # which has not been visited yet
                            self.visited.append(stopping_position)  # Mark this junction visited
                            self.visit(
                                stopping_position)  # Make a recursive call to the function if that cell hasn't been visited yet
                        else:
                            pass
        return

    def set_stopping_pointers(self):
        """ 
        Initializes values for all traffic lights at the computed stopping points
        """
        for _ in self.actual_stopping_positions:
            self.permanent_pointer_position.append(0)  # Permanent pointer position moves by one at each time step
            self.temporary_pointer_position.append(
                0)  # Temporary pointer position makes the traffic light smart by checking for incoming traffic
        return

    def stopping_point_find(self, agent_pos, agent_dir):
        """
        Input - Agent Position and Direction
        Returns - Is Agent At Stopping Point, Stopping Cluster, Index within Cluster
        """
        agent_info = (agent_pos, agent_dir)
        for stopping_cluster in range(len(self.actual_stopping_positions)):
            for stopping_point in self.actual_stopping_positions[stopping_cluster]:
                if stopping_point == agent_info:
                    return True, stopping_cluster, self.actual_stopping_positions[stopping_cluster].index(
                        stopping_point)
        return False, None, None

    def junction_find(self, agent_pos):
        for cluster in range(len(self.actual_junction_cluster)):
            for point in self.actual_junction_cluster[cluster]:
                if point == agent_pos:
                    return True, cluster, self.actual_junction_cluster[cluster].index(point)
        return False, None, None

    def get_traffic_signal(self, agent_pos, agent_dir, others, handle):
        """
        Input - Agent Position, Direction, handle and others 
        Get the traffic signal for an agent
        Returns 1 : If Traffic Signal is Green
        Returns -1 : If Traffic Signal is Red 
        Two clearances are required to be obtained except if an agent is stuck within the cluster
        """

        validity, cluster, index = self.stopping_point_find(agent_pos,
                                                            agent_dir)  # Check whether agent is at Stopping Point
        if validity == False:  # If agent not at stopping point, traffic signal always green
            return 1
        if validity == True:  # If agent is at stopping point
            if cluster in self.stuck_clusters:  # First check if another agent is stuck inside the cluster
                current_pointer_pos = self.permanent_pointer_position[cluster]
                if current_pointer_pos == index:
                    return 1
                clearance = self.get_clearence(cluster, index, others)
                if clearance == True:  # If another agent is stuck, then directly get clearance for the traffic light
                    return 1
                else:
                    return -1

            if cluster in self.timed_out_clusters:
                current_pointer_pos = self.permanent_pointer_position[cluster]
                if current_pointer_pos == index:
                    return 1
                clearance = self.get_clearence(cluster, index, others)
                if clearance == True:  # If another agent is stuck, then directly get clearance for the traffic light
                    return 1
                else:
                    return -1

            # If no agent is stuck inside the cluster, then 2 clearances need to be obtained        
            cleared = self.get_first_clearance(cluster,
                                               handle)  # First clearance checks whether any agent is already occupying the cluster
            if cleared == False:
                return -1

            current_pointer_pos = self.permanent_pointer_position[cluster]
            if current_pointer_pos == index:
                return 1
            else:
                clearance = self.get_clearence(cluster, index,
                                               others)  # Second clearance checks whether other agents are waiting to go inside the cluster,
                if clearance == True:  # If other agents are waiting to go inside cluster, only 1 of them gets a green traffic light
                    return 1
                else:
                    return -1

    def get_agent_stuck(self, cluster, others, position, direction, handle):
        """ 
        Input - Current cluster, others, agent pos, agent dir , agent handle
        Recursively checks whether agent inside a cluster has any possible exit
        """
        # Get the next valid positions
        valid_act_pos_pair = self._get_next_valid_position(position, direction)
        for action, next_pos in valid_act_pos_pair.items():
            if next_pos[1] is not None:
                # If that position,direction has not been checked before
                if (next_pos[1], next_pos[0]) not in self.possible_positions:
                    self.possible_positions.append((next_pos[1], next_pos[0]))
                    # If that position is a stopping point and it is unoccupied , then agent has a free exit
                    if next_pos[1] in self.stopping_positions_only[cluster]:
                        if (next_pos[1], next_pos[0]) not in others.keys():
                            return 0
                    # If that position is another junction whithin the cluster, then recursively call this function        
                    elif next_pos[1] in self.actual_junction_cluster[cluster]:
                        output = self.get_agent_stuck(cluster, others, next_pos[1], next_pos[0], handle)
                        if output == 0:
                            return 0
                    else:
                        pass
        # If all exits are occupied by other agents, then the agent inside the cluster is stuck
        # In such a case, allowing 1 more agent entry into the cluster might be able to solve the problem            
        return 1

    def get_stuck_clusters(self):
        """
        Computes the clusters inside which agents are stuck (no unoccupied agents from that cluster) 
        Returns - Cluster indexes in which agents are stuck
        """
        self.stuck_clusters = []
        others = self.get_others(0,0)  # Get information of all agents
        for cluster in range(len(self.actual_junction_cluster)):  # Iterate over all clusters
            counter = 0
            total = 0
            for handle in range(len(self.env.agents)):  # Iterate over all agents
                position = self.env.agents[handle].position
                direction = self.env.agents[handle].direction
                if position in self.actual_junction_cluster[
                    cluster]:  # If an agent is inside a particular cluster, call function
                    self.possible_positions = []  # to determine whether the agent is stuck
                    self.possible_positions.append(
                        (self.env.agents[handle].position, self.env.agents[handle].direction))
                    stuck = self.get_agent_stuck(cluster, others, position, direction, handle)
                    if stuck:  # If the agent is stuck inside cluster, increment the counter
                        counter += 1
                    total += 1
            if counter == 1 and total < 2:  # Make sure that no more than 2 agents are allowed entry into the cluster
                self.stuck_clusters.append(cluster)
        return self.stuck_clusters

    def get_timed_out_clusters(self):
        self.timed_out_clusters = []
        self.num_agents_in_clusters = [0 for i in range(len(self.actual_junction_cluster))]
        self.timed_out = [0 for i in range(len(self.actual_junction_cluster))]
        for handle in range(len(self.env.agents)):
            if self.env.agents[handle].position != None:
                validity, cluster, index = self.junction_find(self.env.agents[handle].position)
                if validity == True:
                    if self.agent_in_clusters[handle][0] == cluster:
                        self.agent_in_clusters[handle][1] += 1
                    else:
                        self.agent_in_clusters[handle][0] = cluster
                        self.agent_in_clusters[handle][1] = 1

                    self.num_agents_in_clusters[cluster] += 1
                    if self.agent_in_clusters[handle][1] > 10:
                        self.timed_out[cluster] = 1
        for cluster in range(len(self.actual_junction_cluster)):
            if self.num_agents_in_clusters[cluster] < 2 and self.timed_out[cluster] == 1:
                self.timed_out_clusters.append(cluster)
        return self.timed_out_clusters

    def get_others_complete(self):
        """
        Returns tuple of (position,opposite direction) of each agent.
        Unborn/Completed agents are set to have : (-3,-3,0) 
        """
        self.others0 = {} 
        self.others1 ={}
        self.others2 ={}  
        self.others3 ={}  
        self.others4 ={} 
        for id in range(len(self.env.agents)):
            if self.env.agents[id].position is None:
                if self.env.dones[id] is True:
                    otherspos = (-id, -id)
                    othersdir = 0
                    othersdirections =0 
                else:
                    otherspos = (-id, -id)
                    othersdir = 0
                    othersdirections =0 
            else:  # position not None
                otherspos = self.env.agents[id].position
                othersdir = (self.env.agents[id].direction + 2) % 4
                othersdirections = self.env.agents[id].direction 
            self.others0[otherspos, othersdir] = id 
            self.others1[otherspos, othersdirections] = id 
            self.others2[otherspos] = id 
            self.others3[otherspos] = id  
            self.others4[id] = othersdirections 
        return 

    def get_others(self,handle,value) :
        if value ==0 :
            others = copy.copy(self.others0) 
            return others   
        elif value==1 :
            others = copy.copy(self.others1)
            if self.env.agents[handle].position is not None :
                agentpos = self.env.agents[handle].position
                agentdir = self.env.agents[handle].direction 
                del others[agentpos,agentdir] 
            return others 
        elif value ==2 :
            others = copy.copy(self.others2) 
            return others  
        elif value ==3 :
            others = copy.copy(self.others3) 
            if self.env.agents[handle].position is not None :
                agentpos = self.env.agents[handle].position
                del others[agentpos]
            return others 
        else : 
            others = copy.copy(self.others4) 
            del others[handle] 
            return others             

    def get_first_clearance(self, cluster, handle):
        """
        Checks whether a particular cluster is occupied 
        Returns 0 - If Occupied
        Returns 1 - If Free 
        """
        all_handles = [i for i in range(len(self.env.agents))]
        others = self.get_others(handle,3)   
        for value in range(len(self.actual_junction_cluster[cluster])):  # Checks whether the cluster is occupied
            if self.actual_junction_cluster[cluster][value] in others.keys():
                return 0
        return 1

    def get_clearence(self, cluster, index, others):
        """
        Executes smart time-dependent traffic light 
        Returns 1 - If traffic light is green
        Returns 0 - If traffic light is red
        If more than 1 agent is waiting to enter a cluster, only a single agent gets a clearance
        If only 1 agent waiting to enter a cluster, clearance always green for that agent 
        Pointers change values at each timestep
        """

        for positions in range(
                len(self.actual_stopping_positions[cluster])):  # Iterate over all stopping positions of a cluster
            if self.actual_stopping_positions[cluster][self.permanent_pointer_position[cluster]] in others.keys():
                return 0,  # Check if the position pointed by permanent pointer is occupied by another agent
            if self.temporary_pointer_position[cluster] == index:
                self.temporary_pointer_position[cluster] = self.permanent_pointer_position[cluster]
                return 1  # Check if temporary pointer index is the same as our agent's index, if yes return 1 (green)
            else:
                self.temporary_pointer_position[cluster] = (self.temporary_pointer_position[cluster] + 1) % len(
                    self.actual_stopping_positions[cluster])  # Increment temporary pointer to check at next position
                if self.actual_stopping_positions[cluster][self.temporary_pointer_position[cluster]] in others.keys():
                    self.temporary_pointer_position[cluster] = self.permanent_pointer_position[cluster]
                    return 0  # If temporary pointer index is same as another agent's index , then traffic signal for our agent is red

    def update_pointers(self):
        """
        Updates pointer by 1  at each timestep 
        """
        for i in range(len(self.actual_stopping_positions)):
            self.permanent_pointer_position[i] = (self.time) % len(self.actual_stopping_positions[i])
            self.temporary_pointer_position[i] = (self.time) % len(self.actual_stopping_positions[i])
        return

#    def get_others_traffic_light(self, handle):
#        """
#        Compute positions of all agents except our agent 
#        Returns - Others
#        """
#        all_handles = [i for i in range(len(self.env.agents))]
#        others = []
#        others2 = []
#        for id in all_handles:
#            if id != handle:
#                if self.env.agents[id].position is None:
#                    if self.env.dones[id] is True:
##                        otherspos = (-3, -3)
 #                       othersdirections = 0
 #                   else:
 #                       otherspos = (-3, -3)
 #                       othersdirections = self.env.agents[id].initial_direction
 #               else:  # position not None
 #                   otherspos = self.env.agents[id].position
 #                   othersdirections = self.env.agents[id].direction
 #               others.append((otherspos, othersdirections))
 #               others2.append([otherspos, othersdirections])
 #       return others, others2

#    def get_initial_positions(self):
#        """
#        Get initial position of all agents 
#        """
#        all_handles = [i for i in range(len(self.env.agents))]
#        for id in all_handles:
#            self.agent_initial_positions.append(
#                [id, self.env.agents[id].initial_position, self.env.agents[id].initial_direction,
#                 self.env.agents[id].target])
#        return

    def get_initialization_queue(self):
        for id in range(len(self.env.agents)):
            info = (self.env.agents[id].initial_position, self.env.agents[id].initial_direction,
                    self.env.agents[id].target)
            if info in self.queues.keys():
                self.queues[info][0] += 1
                self.queues[info].append(id)
            else:
                self.queues[info] = [1]
                self.queues[info].append(id)
        queue = sorted(self.queues.values(), key=itemgetter(0), reverse=True)
        self.queue = []
        for element in queue:
            self.queue.append([0, element[1:]])
        return

    def get_initialization(self):
        clusters_to_check = []
        queue_size = 0
        if (self.num_active_agents.count(1) > self.upper_bound) and (self.initialization_timestep < self.max_timestep):
            return
        if len(self.agents_activated) == len(self.env.agents):
            return
        for index_cluster in range(len(self.queue)):
            if index_cluster not in self.clusters_activated:
                queue_size += len(self.queue[index_cluster][1])
                clusters_to_check.append(index_cluster)
            if self.upper_bound < queue_size:
                break
        for index in clusters_to_check:
            if self.num_active_agents.count(1) > self.upper_bound:
                break
            for agent in self.queue[index][1]:
                if agent not in self.agents_activated:
                    others = self.get_others(agent,3 )
                    allowed = self.initialize(agent, self.observations[agent], others)
                    if allowed:
                        self.agents_activated.append(agent)
                        self.queue[index][0] += 1
                        self.num_active_agents[agent] = 1
                        self.observations[agent][self.OBS_SIZE - self.ADDITIONAL_INPUT + 1] = 0
                        if self.queue[index][0] == len(self.queue[index][1]):
                            self.clusters_activated.append(index)
                        self.initialization_timestep = 0
                        break
        if self.initialization_timestep > self.max_timestep:
            for index_cluster in range(len(self.queue)):
                if index_cluster not in self.clusters_activated :
                    for agent in self.queue[index_cluster][1]:
                        if agent not in self.agents_activated:
                            others = self.get_others(agent,3 )
                            allowed = self.initialize(agent, self.observations[agent], others)
                            if allowed:
                                self.num_active_agents[agent] = 1
                                self.agents_activated.append(agent)
                                self.queue[index_cluster][0] += 1
                                self.observations[agent][self.OBS_SIZE - self.ADDITIONAL_INPUT + 1] = 0
                                if self.queue[index_cluster][0] == len(self.queue[index_cluster][1]):
                                    self.clusters_activated.append(index_cluster)
                                self.initialization_timestep = 0
                                break
                    if allowed:
                        break

        return

    def initialize(self, handle, obs, others):
        """
        Input - Observation, agent handle, others
        Returns 1 if an agent is not allowed to initialize
        Returns 0 if an agent is allowed to initialize 
        """
        # An agent is only allowed to initialize if - 
        # 1) No other unborn agent with a smaller handle is currently at the same position waiting to be born
        # 2) No currently alive agent is at the agent's position
        # 3) No currently alive agent is currently behind the agent , this would lead to a crash
        # 4) The agent won't be blocked if it initializes 

        for agent in range(len(self.env.agents)):
            if (agent != handle) and (agent in self.agents_activated) and (self.env.agents[agent].position == None and \
                                                                           self.num_active_agents[agent] != 2) and (
                    self.env.agents[agent].initial_position == self.env.agents[handle].initial_position):
                return 0

        
        if (self.env.agents[handle].initial_position in others.keys()):
                return 0
        valid_act_pos_pair = self._get_next_valid_position(self.env.agents[handle].initial_position,
                                                           ((self.env.agents[handle].initial_direction + 2) % 4))
        for action, pos in valid_act_pos_pair.items():
            if pos[1] is not None:
                    if (pos[1] in others.keys()):
                        return 0
        agentblocking = obs[self.ENTRYS_PER_COLUMN + 2]
        if agentblocking != 0:
            return 0
        # if (self.num_active_agents.count(1) < self.upper_bound) or (self.initialization_timestep>self.max_timestep) : 
        #     if (self.initialization_timestep>self.max_timestep) :
        #         self.initialization_timestep = 0 
        #     self.num_active_agents[handle] = 1     
        #     return 0 
        return 1


if __name__ == '__main__':
    from flatland.envs.rail_env import RailEnv
    from flatland.envs.rail_generators import sparse_rail_generator
    from flatland.envs.line_generators import sparse_line_generator
    from flatland.utils.rendertools import RenderTool

    num_agent = 20


    def Complex_params():
        grid_width = 25  # min(int(np.random.uniform(ENVIRONMENT_SIZE[0], ENVIRONMENT_SIZE[1] )),

        # int(np.random.uniform(ENVIRONMENT_SIZE[0], ENVIRONMENT_SIZE[1] )))
        grid_height = 25  # min(int(np.random.uniform(ENVIRONMENT_SIZE[0], ENVIRONMENT_SIZE[1])),

        # nt(np.random.uniform(ENVIRONMENT_SIZE[0], ENVIRONMENT_SIZE[1] )))
        rnd_start_goal = 8 + np.random.randint(0,
                                               3)  # int(np.random.uniform(num_workers, num_workers+1+episode_difficulty ))
        # int(np.random.uniform( num_workers , min(grid_width,grid_height))),

        rnd_extra = 1  # int(np.random.uniform(0 , 1+2*episode_difficulty ))
        # int(np.random.uniform( 0 , min(grid_width,grid_height))))
        rnd_min_dist = int(
            0.6 * min(grid_height, grid_width))  # int(np.random.uniform( episode_difficulty , 4+2*episode_difficulty ))
        rnd_max_dist = 99999  # int(np.random.uniform(3+episode_difficulty, 6+2*episode_difficulty))
        rnd_seed = 3

        return grid_width, grid_height, rnd_start_goal, rnd_extra, rnd_min_dist, rnd_max_dist, rnd_seed


    grid_width, grid_height, rnd_start_goal, rnd_extra, rnd_min_dist, rnd_max_dist, rnd_seed = Complex_params()
    gameEnv = RailEnv(width=35, height=20,
                      rail_generator=sparse_rail_generator(
                          max_num_cities=5,
                          max_rails_between_cities=2,
                          max_rails_in_city=3,
                          grid_mode=False,
                          seed=rnd_seed),
                      line_generator=sparse_line_generator(),
                      obs_builder_object=StateMaskingObs(),
                      number_of_agents=num_agent)
    gameEnv.reset()
    env_renderer = RenderTool(gameEnv)

    for t in range(500):
        gameEnv.reset()
        print(t) 
        for i in range(30):
            step_result = gameEnv.step({0: 2, 1: 2, 2: 2, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2})
            obs = step_result[0]
            #print(obs)
            env_renderer.render_env(show=True, frames=True, show_observations=True)
            # print(obs)
            # print(len(obs[0]))
            #input()


In [ ]:
%%writefile agents/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeepMLP(nn.Module):
    """
    Standard 7-layer MLP structure consistent with NeurIPS 2020 architecture.
    """
    def __init__(self, in_features, out_features):
        super(DeepMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1024), nn.ReLU(),
            nn.Linear(1024, out_features)
        )

    def forward(self, x):
        return self.net(x)

class ACNet(nn.Module):
    def __init__(self, state_size, action_size, n_agents):
        """
        Multi-Agent Actor-Critic Network
        - Single Critic with two value functions (Centralized)
        - Multiple Actors using the DeepMLP structure (Decentralized)
        - Multiple Action Estimators using the same DeepMLP structure (Decentralized)
        """
        super(ACNet, self).__init__()
        self.n_agents = n_agents
        self.state_size = state_size
        self.action_size = action_size

        # --- CENTRALIZED CRITIC ---
        self.critic_net = nn.Sequential(
            nn.Linear(state_size * n_agents, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.value_head = nn.Linear(256, 1)

        # --- SHARED ACTOR ---
        # Single actor network shared across all homogeneous agents
        self.actor = DeepMLP(state_size, action_size)

        # --- SHARED ESTIMATOR ---
        self.estimator = DeepMLP(state_size, action_size)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=nn.init.calculate_gain('relu'))
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        
        # Adjust head gains for stability
        nn.init.orthogonal_(self.value_head.weight, gain=1.0)
        
        nn.init.orthogonal_(self.actor.net[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.estimator.net[-1].weight, gain=0.01)

    def forward(self, x, centralized_state=None):
        """
        Args:
            x (torch.Tensor): Tensor of individual states [batch, n_agents, state_size]
            centralized_state (torch.Tensor, optional): Centralized joint state [batch, n_agents * state_size]
        Returns:
            policies (torch.Tensor): Agent policy outputs [batch, n_agents, action_size]
            estimations (torch.Tensor): Agent estimation outputs [batch, n_agents, action_size]
            values (torch.Tensor): Value output [batch, 1]
        """
        batch_size = x.size(0)
        # 1. Process all agent states through the shared Actor
        # x is [batch_size, n_agents, state_size], Linear layers broadcast the computation automatically!
        actor_logits = self.actor(x) 
        policies = F.softmax(actor_logits, dim=-1) # [batch_size, n_agents, action_size]

        # Valid action estimation
        estimator_logits = self.estimator(x) # [batch_size, n_agents, action_size]

        # Centralized Critic
        if centralized_state is None:
            centralized_state = x.view(batch_size, -1)
            
        v = self.value_head(self.critic_net(centralized_state))
        
        return policies, estimator_logits, v


In [ ]:
%%writefile Training.py
"""
Training.py  —  GPU-optimised PPO for Flatland MARL
=====================================================

Architecture:
  - 11 CPU worker processes  (mp.Process, spawn)  — each owns one Flatland env
  - 1  main process          (GPU)                 — batched inference + PPO update
  - Communication via mp.Pipe (worker→main: obs/rew/done, main→worker: actions)

Why spawn (not fork):
  - Main process initialises CUDA before workers start.
  - fork() copies the CUDA context into child processes → corrupts it.
  - spawn() starts a fresh Python interpreter per worker → always safe.

Why workers build their own env (not passed from main):
  - Flatland's StateMaskingObs uses copy.deepcopy on attr-slotted objects.
  - These cannot be pickled/sent across process boundaries.
  - This was the original crash: KeyboardInterrupt trace ending in
    slots_getstate inside DistToNextJunction → deepcopy → mp pipe send.
  - Fix: each worker calls make_env(seed_offset=rank) itself.

GPU optimisations:
  - 11 envs step in parallel on CPU while main does PPO backward.
  - All agents × all workers batched into one tensor → one GPU forward pass.
  - Rollout buffer pre-allocated on GPU — no repeated CPU→GPU copies.
"""

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.distributions import Categorical

from flatland.envs.rail_env import RailEnv
from flatland.envs.rail_generators import sparse_rail_generator
from flatland.envs.line_generators import SparseLineGen
from flatland.envs.malfunction_generators import ParamMalfunctionGen, MalfunctionParameters

from config import Config
from NewAgentInitObs import StateMaskingObs
from agents.models import ACNet
from shortestpath import clear_trans_cache   # clear shared transition cache on env reset


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

N_WORKERS    = 11       # 12 Colab L4 cores - 1 for main process
ROLLOUT_LEN  = 128      # steps collected per worker before PPO update
PPO_EPOCHS   = 4
PPO_CLIP     = 0.2
ENTROPY_COEF = 0.01
GAMMA        = 0.99
LR           = 3e-4
MAX_EPISODES = 1000
SAVE_EVERY   = 20
A_SIZE       = 4        # internal: 0=Left, 1=Forward, 2=Right, 3=Stop

_SHUTDOWN = "SHUTDOWN"
_RESET    = "RESET"


# ---------------------------------------------------------------------------
# Environment factory  (called inside worker processes only)
# ---------------------------------------------------------------------------

def make_env(seed_offset: int = 0) -> RailEnv:
    rail_gen = sparse_rail_generator(
        max_num_cities=Config.n_cities,
        max_rails_between_cities=Config.max_rails_between_cities,
        max_rail_pairs_in_city=Config.max_rail_pairs_in_city,
        seed=Config.seed + seed_offset,
        grid_mode=False,
    )
    malfunction_gen = ParamMalfunctionGen(MalfunctionParameters(
        malfunction_rate=Config.malfunction_rate,
        min_duration=Config.min_malfunction_duration,
        max_duration=Config.max_malfunction_duration,
    ))
    env = RailEnv(
        width=Config.width,
        height=Config.height,
        rail_generator=rail_gen,
        line_generator=SparseLineGen(seed=None),
        number_of_agents=Config.n_agents,
        obs_builder_object=StateMaskingObs(),
        malfunction_generator=malfunction_gen,
    )
    return env


# ---------------------------------------------------------------------------
# Expert heuristics  (called inside env_worker — env is local to that process)
# ---------------------------------------------------------------------------

def state_classifier(env, pos, direction, _cache=None):
    """
    Classify current cell:
      0: Normal  1: Stopping Point  2: Junction  3: Must Stop  4: Stopping+Must Stop
    Uses an optional dict cache to avoid repeated rail.get_transitions() calls.
    """
    key = (pos, direction)
    if _cache is not None and key in _cache:
        avb_moves = _cache[key]
    else:
        avb_moves = env.rail.get_transitions((pos, direction))
        if _cache is not None:
            _cache[key] = avb_moves
    if sum(avb_moves) > 1:
        return 2
    avb_idx     = avb_moves.index(1)
    dir_offsets = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    next_pos    = (pos[0] + dir_offsets[avb_idx][0],
                   pos[1] + dir_offsets[avb_idx][1])
    is_next_junction = False
    for d in range(4):
        nkey = (next_pos, d)
        if _cache is not None and nkey in _cache:
            trans = _cache[nkey]
        else:
            trans = env.rail.get_transitions((next_pos, d))
            if _cache is not None:
                _cache[nkey] = trans
        if sum(trans) > 2:
            is_next_junction = True
            break
    occupied = any(a.position == next_pos for a in env.agents)
    if is_next_junction:
        return 4 if occupied else 1
    return 3 if occupied else 0


def get_valid_actions(env, obs, agent_id, _cache=None):
    """
    Port of _NextValidActions from the NeurIPS reference code.
    Returns a list of valid internal actions: 0=Left, 1=Forward, 2=Right, 3=Stop.
    _cache: optional dict for caching rail.get_transitions results.
    """
    traffic_signal = obs[24]
    homo_junctions = obs[27:30]
    if traffic_signal == -1:
        return [3]
    currentobs  = np.reshape(obs[0:24], (3, -1))
    agent       = env.agents[agent_id]
    actual_pos  = agent.position  if agent.position  is not None else agent.initial_position
    actual_dir  = agent.direction if agent.direction is not None else agent.initial_direction
    state = state_classifier(env, actual_pos, actual_dir, _cache)
    if state in [3, 4]:
        return [3]
    elif state == 0:
        return [1]
    elif state == 1:
        SolExist         = [currentobs[0][0], currentobs[1][0], currentobs[2][0]]
        agentsblocking   = [currentobs[0][2], currentobs[1][2], currentobs[2][2]]
        agentsblockjunct = [currentobs[0][3], currentobs[1][3], currentobs[2][3]]
        for i in range(3):
            if SolExist[i] == 1 and agentsblocking[i] == 0:
                return [1, 3]
        if int(np.sum(homo_junctions == 1)) >= 2:
            for i in range(3):
                if SolExist[i] == 1 and agentsblockjunct[i] == 0 and homo_junctions[i] == 1:
                    return [1, 3]
        return [3]
    else:  # Junction
        SolExist         = [currentobs[0][0], currentobs[1][0], currentobs[2][0]]
        agentsblocking   = [currentobs[0][2], currentobs[1][2], currentobs[2][2]]
        agentsblockjunct = [currentobs[0][3], currentobs[1][3], currentobs[2][3]]
        validactions = []
        for i in range(3):
            if SolExist[i] == 1 and agentsblocking[i] == 0:
                validactions.append(i)
        if validactions:
            return validactions
        if int(np.sum(homo_junctions == 1)) >= 2:
            for i in range(3):
                if SolExist[i] == 1 and agentsblockjunct[i] == 0 and homo_junctions[i] == 1:
                    validactions.append(i)
                    break
            if validactions:
                return validactions
        return [3]


# ---------------------------------------------------------------------------
# Worker process  (pure CPU — no CUDA, no network)
# ---------------------------------------------------------------------------

def env_worker(rank: int, obs_pipe, act_pipe):
    """
    Owns one Flatland env. Protocol:
      Startup  : sends (obs [n_agents, obs_size], valid_tgts [n_agents, A_SIZE])
      Loop     : recvs action dict {agent_id: flatland_action (int)}
                 sends (obs, rewards, done_dict, valid_tgts)
      _RESET   : resets env, sends (obs, valid_tgts)
      _SHUTDOWN: exits
    """
    env    = make_env(seed_offset=rank)
    n      = Config.n_agents
    obs_sz = Config.obs_size
    _trans_cache: dict = {}   # per-episode rail transition cache (pos,dir) -> transitions

    def pack_obs(obs_dict):
        arr = np.zeros((n, obs_sz), dtype=np.float32)
        for i in range(n):
            if obs_dict.get(i) is not None:
                arr[i] = obs_dict[i]
        return arr

    def compute_valid_tgts(obs_dict):
        """Multi-hot [n_agents, A_SIZE] mask of expert-valid actions."""
        vtgts = np.zeros((n, A_SIZE), dtype=np.float32)
        for i in range(n):
            if obs_dict.get(i) is not None:
                try:
                    valid = get_valid_actions(env, obs_dict[i], i, _trans_cache)
                    vtgts[i, valid] = 1.0
                except Exception:
                    vtgts[i, 3] = 1.0   # fallback: stop
            else:
                vtgts[i, 3] = 1.0       # agent done/missing
        return vtgts

    obs_dict, _ = env.reset()
    env._max_episode_steps = Config.max_episode_steps   # override timetable value
    obs_pipe.send((pack_obs(obs_dict), compute_valid_tgts(obs_dict)))

    while True:
        msg = act_pipe.recv()

        if msg == _SHUTDOWN:
            break

        if msg == _RESET:
            _trans_cache.clear()    # rail regenerates — old transitions invalid
            clear_trans_cache()     # also clear shortestpath.py module cache
            obs_dict, _ = env.reset()
            env._max_episode_steps = Config.max_episode_steps   # override timetable value
            obs_pipe.send((pack_obs(obs_dict), compute_valid_tgts(obs_dict)))
            continue

        # Normal env step
        result   = env.step(msg)
        obs_dict = result[0]
        rew_dict = result[1]
        term     = result[2]
        trunc    = result[3] if len(result) > 3 else {}

        obs  = pack_obs(obs_dict)
        rews = np.array([rew_dict.get(i, 0.0) for i in range(n)], dtype=np.float32)

        done = {k: term.get(k, False) or trunc.get(k, False) for k in term}
        done['__all__'] = all(done.get(i, False) for i in range(n))

        obs_pipe.send((obs, rews, done, compute_valid_tgts(obs_dict)))


# ---------------------------------------------------------------------------
# Rollout buffer  (pre-allocated on GPU)
# ---------------------------------------------------------------------------

class RolloutBuffer:
    """Shape: [T, W, A, ...] — T=timesteps, W=workers, A=agents."""

    def __init__(self, T, W, A, obs_sz, act_sz, device):
        self.T      = T
        self.device = device
        kw = dict(device=device)
        self.obs        = torch.zeros(T, W, A, obs_sz, **kw)
        self.actions    = torch.zeros(T, W, A,          dtype=torch.long, **kw)
        self.rewards    = torch.zeros(T, W, A,          **kw)
        self.dones      = torch.zeros(T, W,             **kw)
        self.valid_tgts = torch.zeros(T, W, A, act_sz,  **kw)
        self.log_probs  = torch.zeros(T, W, A,          **kw)
        self.values     = torch.zeros(T, W,             **kw)
        self.ptr = 0

    def store(self, obs, actions, rewards, dones, valid_tgts, log_probs, values):
        t = self.ptr
        self.obs[t]        = obs
        self.actions[t]    = actions
        self.rewards[t]    = rewards
        self.dones[t]      = dones
        self.valid_tgts[t] = valid_tgts
        self.log_probs[t]  = log_probs
        self.values[t]     = values
        self.ptr += 1

    def reset_ptr(self):
        self.ptr = 0

    def compute_returns(self, bootstrap: torch.Tensor) -> torch.Tensor:
        """Monte-Carlo returns. bootstrap: [W]. Returns [T, W]."""
        returns       = torch.zeros(self.T, bootstrap.shape[0], device=self.device)
        joint_rewards = self.rewards.sum(dim=-1)  # [T, W]
        R = bootstrap.clone()
        for t in reversed(range(self.T)):
            R          = joint_rewards[t] + GAMMA * R * (1.0 - self.dones[t])
            returns[t] = R
        return returns


# ---------------------------------------------------------------------------
# PPO update  (everything on GPU)
# ---------------------------------------------------------------------------

def ppo_update(network, optimizer, buffer, returns):
    T, W, A, obs_sz = buffer.obs.shape

    flat_obs      = buffer.obs.view(T * W, A, obs_sz)
    flat_acts     = buffer.actions.view(T * W, A)
    flat_returns  = returns.view(T * W)
    flat_tgts     = buffer.valid_tgts.view(T * W, A, -1)
    flat_lp_old   = buffer.log_probs.view(T * W, A)
    flat_vals_old = buffer.values.view(T * W)

    adv = flat_returns - flat_vals_old
    if adv.numel() > 1 and adv.std() > 1e-8:
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    total_loss = 0.0
    for _ in range(PPO_EPOCHS):
        policies, estimations, v = network(flat_obs)
        cur_v = v.squeeze(-1)

        # Actor loss (PPO clip)
        act_p  = policies.gather(2, flat_acts.unsqueeze(-1)).squeeze(-1)
        log_p  = torch.log(act_p + 1e-8)
        ratio  = torch.exp(log_p - flat_lp_old)
        adv_e  = adv.unsqueeze(1)
        surr1  = ratio * adv_e
        surr2  = torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * adv_e
        a_loss = -torch.min(surr1, surr2).mean()

        # Critic loss
        std    = flat_returns.std().clamp(min=1e-8)
        r_norm = (flat_returns - flat_returns.mean()) / std
        v_norm = (cur_v        - flat_returns.mean()) / std
        c_loss = F.mse_loss(v_norm, r_norm)

        # Entropy
        ent    = -(policies * torch.log(policies + 1e-8)).sum(-1).mean()

        # Estimator (valid-action imitation)
        e_loss = F.binary_cross_entropy_with_logits(estimations, flat_tgts)

        loss = a_loss + 0.5 * c_loss - ENTROPY_COEF * ent + 0.5 * e_loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(network.parameters(), 0.5)
        optimizer.step()

    return total_loss / PPO_EPOCHS


# ---------------------------------------------------------------------------
# Main training loop
# ---------------------------------------------------------------------------

def train():
    # ---- Device ----
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device : {device}")
    if device.type == "cuda":
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
        print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Workers: {N_WORKERS}  (cores used: {N_WORKERS + 1} / {os.cpu_count()})")
    print()

    # ---- Network + optimiser ----
    network = ACNet(
        state_size=Config.obs_size,
        action_size=A_SIZE,
        n_agents=Config.n_agents,
    ).to(device)
    optimizer = optim.AdamW(network.parameters(), lr=LR,
                            betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

    # ---- Resume from checkpoint ----
    start_episode = 0
    ckpt_dir = os.path.abspath(Config.checkpoint_dir)
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpts = [f for f in os.listdir(ckpt_dir)
             if f.startswith("checkpoint_") and f.endswith(".pth")]
    if ckpts:
        latest = max(ckpts, key=lambda f: int(f.split("_")[1].split(".")[0]))
        ckpt   = torch.load(os.path.join(ckpt_dir, latest), map_location=device)
        network.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_episode = ckpt.get("episode", 0)
        print(f"Resumed: {latest}  (episode {start_episode})")

    # ---- Spawn worker processes ----
    # spawn is set in __main__ block before train() is called.
    # Workers are daemon=True so they die automatically if main crashes.
    workers, obs_pipes, act_pipes = [], [], []
    for rank in range(N_WORKERS):
        p_obs, c_obs = mp.Pipe(duplex=False)   # worker → main  (p_obs=read, c_obs=write)
        p_act, c_act = mp.Pipe(duplex=False)   # main → worker  (p_act=read, c_act=write)
        # Worker needs: c_obs (write end, to send obs) + p_act (read end, to recv actions)
        # Main  needs: p_obs (read end, to recv obs)  + c_act (write end, to send actions)
        p = mp.Process(target=env_worker,
                       args=(rank, c_obs, p_act),
                       daemon=True)
        p.start()
        workers.append(p)
        obs_pipes.append(p_obs)
        act_pipes.append(c_act)
    print(f"Spawned {N_WORKERS} env workers.")

    # Collect initial obs + valid-action masks from each worker
    init_data     = [obs_pipes[w].recv() for w in range(N_WORKERS)]
    obs_np        = np.stack([d[0] for d in init_data])   # [W, A, obs]
    valid_tgts_np = np.stack([d[1] for d in init_data])   # [W, A, A_SIZE]
    dones  = [{'__all__': False, **{i: False for i in range(Config.n_agents)}}
              for _ in range(N_WORKERS)]

    # ---- Rollout buffer (GPU) ----
    buffer = RolloutBuffer(
        T=ROLLOUT_LEN, W=N_WORKERS, A=Config.n_agents,
        obs_sz=Config.obs_size, act_sz=A_SIZE, device=device,
    )

    episode_count = start_episode
    ep_reward_buf = [0.0] * N_WORKERS   # running reward sum per worker, reset each episode
    t_start       = time.time()
    print(f"Training: episodes {start_episode} → {MAX_EPISODES}\n")

    # ==========================================================================
    # Main loop
    # ==========================================================================
    while episode_count < MAX_EPISODES:
        buffer.reset_ptr()
        rollout_ep_rewards: list[float] = []   # EpRewards of episodes completed this rollout
        network.eval()

        # ----------------------------------------------------------------------
        # ROLLOUT COLLECTION
        # Key timing:
        #   1. GPU forward pass  (~ms)
        #   2. Send actions to all workers simultaneously
        #   3. Workers step envs IN PARALLEL on their cores  (~seconds, the bottleneck)
        #   4. Collect results
        # Step 3 overlaps with any CPU work we do between send and recv.
        # ----------------------------------------------------------------------
        with torch.no_grad():
            for _ in range(ROLLOUT_LEN):

                # 1. Batched GPU inference — all workers × all agents in one call
                obs_t              = torch.from_numpy(obs_np).to(device)      # [W, A, obs]
                policies, _, v     = network(obs_t)                            # [W, A, act], [W,1]
                values_t           = v.squeeze(-1)                             # [W]

                # 2. Sample actions with expert valid-action masking (CPU, fast)
                actions_np    = np.zeros((N_WORKERS, Config.n_agents), dtype=np.int64)
                actions_dicts = [{} for _ in range(N_WORKERS)]

                policies_cpu = policies.cpu()   # one transfer for all workers
                for w in range(N_WORKERS):
                    for i in range(Config.n_agents):
                        if dones[w].get(i, False):
                            actions_np[w, i]    = 3
                            actions_dicts[w][i] = 4   # Flatland stop
                            continue
                        probs = policies_cpu[w, i].clone().clamp(min=0)
                        # Apply expert hard mask
                        mask  = torch.from_numpy(valid_tgts_np[w, i])
                        probs = probs * mask
                        if torch.isnan(probs).any() or probs.sum() <= 0:
                            probs = mask.clone()
                            if probs.sum() <= 0:
                                probs[3] = 1.0   # force stop
                        probs /= probs.sum()
                        act = Categorical(probs=probs).sample().item()
                        actions_np[w, i]    = act
                        # internal 0=L,1=F,2=R,3=Stop → Flatland 1=L,2=F,3=R,4=Stop
                        actions_dicts[w][i] = act + 1 if act < 3 else 4

                # 3. Send actions to ALL workers — they start stepping in parallel NOW
                for w in range(N_WORKERS):
                    act_pipes[w].send(actions_dicts[w])

                # (Workers are running env.step() on their CPU cores in parallel here)

                # Compute log-probs on GPU while workers are busy
                actions_t = torch.from_numpy(actions_np).to(device)           # [W, A]
                act_probs = policies.gather(2, actions_t.unsqueeze(-1)).squeeze(-1)
                log_probs = torch.log(act_probs + 1e-8)                       # [W, A]
                valid_t   = torch.from_numpy(valid_tgts_np).to(device)        # current-state mask

                # 4. Collect results — blocks until all 11 workers finish
                results          = [obs_pipes[w].recv() for w in range(N_WORKERS)]
                next_obs_np      = np.stack([r[0] for r in results])          # [W, A, obs]
                rewards_np       = np.stack([r[1] for r in results])          # [W, A]
                next_dones       = [r[2]            for r in results]
                valid_tgts_np    = np.stack([r[3] for r in results])          # [W, A, A_SIZE] next-state masks

                rewards_t = torch.from_numpy(rewards_np).to(device)
                dones_t   = torch.tensor(
                    [float(next_dones[w]['__all__']) for w in range(N_WORKERS)],
                    device=device,
                )

                buffer.store(obs_t, actions_t, rewards_t, dones_t,
                             valid_t, log_probs, values_t)

                # ---- Inline reset + per-episode logging ----
                # Both loops merged so we only iterate N_WORKERS once.
                for w in range(N_WORKERS):
                    if next_dones[w]['__all__']:
                        episode_count += 1
                        rollout_ep_rewards.append(ep_reward_buf[w])
                        elapsed = time.time() - t_start
                        print(f"  [W{w:02d}] Ep {episode_count:4d}/{MAX_EPISODES} | "
                              f"EpReward {ep_reward_buf[w]:8.2f} | "
                              f"{elapsed/60:.1f}min")
                        ep_reward_buf[w] = 0.0   # reset accumulator

                        act_pipes[w].send(_RESET)
                        reset_data        = obs_pipes[w].recv()
                        next_obs_np[w]    = reset_data[0]
                        valid_tgts_np[w]  = reset_data[1]
                        next_dones[w]     = {'__all__': False,
                                             **{i: False for i in range(Config.n_agents)}}

                # Accumulate per-worker episode rewards (after reset so done workers start fresh)
                for w in range(N_WORKERS):
                    ep_reward_buf[w] += float(rewards_np[w].sum())

                obs_np = next_obs_np
                dones  = next_dones

        # ----------------------------------------------------------------------
        # PPO UPDATE  (GPU — workers idle during this, waiting for next actions)
        # ----------------------------------------------------------------------
        network.train()
        with torch.no_grad():
            obs_t = torch.from_numpy(obs_np).to(device)
            _, _, v_boot = network(obs_t)
            bootstrap    = v_boot.squeeze(-1)          # [W]

        returns  = buffer.compute_returns(bootstrap)   # [T, W]
        avg_loss = ppo_update(network, optimizer, buffer, returns)

        # ----------------------------------------------------------------------
        # LOGGING
        # ----------------------------------------------------------------------
        elapsed = time.time() - t_start
        if rollout_ep_rewards:
            ep_reward_str = f"EpReward {sum(rollout_ep_rewards)/len(rollout_ep_rewards):8.2f} (n={len(rollout_ep_rewards)})"
        else:
            # No episode finished this rollout — fall back to per-step mean
            ep_reward_str = f"Reward/step {buffer.rewards.sum(dim=-1).mean().item():.3f} (no ep done)"
        print(f"Ep {episode_count:4d}/{MAX_EPISODES} | "
              f"Loss {avg_loss:.4f} | "
              f"{ep_reward_str} | "
              f"{elapsed/60:.1f}min elapsed")

        # ----------------------------------------------------------------------
        # CHECKPOINT
        # ----------------------------------------------------------------------
        if episode_count % SAVE_EVERY == 0 and episode_count > start_episode:
            path = os.path.join(ckpt_dir, f"checkpoint_{episode_count}.pth")
            torch.save({
                "model_state_dict":     network.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "episode":              episode_count,
            }, path)
            print(f"  >> Saved {path}")

        # (Workers are reset inline during the rollout — nothing to do here.)

    # ==========================================================================
    # SHUTDOWN
    # ==========================================================================
    for w in range(N_WORKERS):
        act_pipes[w].send(_SHUTDOWN)
    for p in workers:
        p.join(timeout=10)

    final = os.path.join(ckpt_dir, "appo_final.pth")
    torch.save({
        "model_state_dict":     network.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "episode":              episode_count,
    }, final)
    print(f"\nDone. Saved → {final}")
    print(f"Total time: {(time.time() - t_start) / 60:.1f} min")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Must be set before ANY CUDA call and before spawning.
    # force=True in case something imported above already set it.
    mp.set_start_method("spawn", force=True)
    train()


### 4. Run Training


In [ ]:
!python Training.py